# Qwen2.5-3B Mega-Calibration (Colab)

**One Colab run → calibration data for 4 downstream dismantle wins:**
1. **Eagle5 v2 head training** (layer-32 residual + intermediate)
2. **AWQ smoothing factors** (per-channel mean|x| × all layers × 7 sites)
3. **Per-channel W4A8 static scales** (per-channel max|x| × all layers × 7 sites)
4. **Quality benchmarks ground truth** (top-100 logits per token)

**Compute:** ~6-8 hr on H100, ~$8-12 in Pro compute units. Fits monthly Pro budget.

**Output:** ~3-5 GB of parquet shards on Drive + 1 small activation-stats .npz file.

In [ ]:
# Cell 1 — Mount Drive + verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'No CUDA — set Runtime → Change runtime type → GPU'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  VRAM: {vram:.1f} GB')

In [ ]:
# Cell 2 — Install deps
!pip install -q 'accelerate>=1.0' 'datasets>=3.0' 'pyarrow>=17' 'tqdm>=4.66' 'zstandard' 'bitsandbytes>=0.43'
import transformers, datasets, pyarrow, accelerate
print(f'transformers={transformers.__version__}  datasets={datasets.__version__}')
print(f'pyarrow={pyarrow.__version__}  accelerate={accelerate.__version__}')

In [ ]:
# Cell 3 — Clone dismantle
import os
if not os.path.exists('/content/dismantle'):
    !cd /content && git clone --depth 1 https://github.com/joshuahickscorp/dismantle.git
else:
    !cd /content/dismantle && git pull --ff-only
%cd /content/dismantle
!ls -l colab/mega_calibrate.py

In [ ]:
# Cell 4 — RUN MEGA-CALIBRATION
#
# Qwen2.5-3B-Instruct (dismantle's primary product target).
# 2000 prompts × 2048 tokens, capture-layer=32 (of 36, near-top).
#
# Auto-detects GPU:
#   >=35 GB (A100/H100): native fp16
#   <35 GB (L4/V100/T4): 4-bit nf4 via bitsandbytes
#
# Resumable: rerun the cell after any disconnect — corpus_simple.py-style
# resume by shard count is built into mega_calibrate.py.

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

OUT = '/content/qwen3b_corpus'  # local SSD; Cell 5 syncs to Drive
os.makedirs(OUT, exist_ok=True)

import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram >= 35:
    load_flag = ''
    batch = 8 if vram >= 70 else 6
else:
    load_flag = '--load-4bit'
    batch = 4
print(f'VRAM={vram:.0f} GB → batch={batch}  load={"4-bit" if load_flag else "fp16"}')

!python colab/mega_calibrate.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --max-sequences 2000 \
  --max-tokens 2048 \
  --batch-size {batch} \
  --capture-layer 32 \
  --topk-logits 100 \
  --shard-size 16 \
  {load_flag} \
  --out {OUT}

In [ ]:
# Cell 5 — Sync corpus + activation stats → Drive
import os, glob, shutil, time
SRC = '/content/qwen3b_corpus'
DST = '/content/drive/MyDrive/dismantle/qwen3b_corpus'
os.makedirs(DST, exist_ok=True)

for f in sorted(glob.glob(f'{SRC}/*')):
    name = os.path.basename(f)
    d = f'{DST}/{name}'
    if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(f):
        shutil.copy2(f, d)

shards = sorted(glob.glob(f'{DST}/*.parquet'))
total_mb = sum(os.path.getsize(f) for f in shards) / 1e6
stats_path = f'{DST}/per_site_activation_stats.npz'
stats_mb = os.path.getsize(stats_path) / 1e6 if os.path.exists(stats_path) else 0

print(f'\n✅ {len(shards)} shards on Drive, {total_mb:.0f} MB')
print(f'✅ activation stats: {stats_mb:.1f} MB at {stats_path}')
print()
print('Download to laptop:')
print('  1. drive.google.com → MyDrive/dismantle/qwen3b_corpus → Download')
print('  2. Unzip into artifacts/calibration/qwen3b_corpus/ on laptop')
print('  3. Run laptop post-processing:')
print('     - Train Qwen3B Eagle5 head (~2 hr MLX)')
print('     - Apply AWQ algorithm to per_site_activation_stats.npz (~30 min)')
print('     - Bench stacked configs')